In [4]:
!pip install -qU langchain langchain-community langgraph wikipedia arxiv semantic_scholar tavily-python

67.02s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
ERROR: Could not find a version that satisfies the requirement semantic_scholar (from versions: none)
ERROR: No matching distribution found for semantic_scholar


In [ ]:
pip install langgraph

In [17]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END, START
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from operator import add

In [ ]:
pip install langchain-openai

In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

In [ ]:
class DeepSearch(TypedDict):
    query: str
    keyword: str
    keywords: list[str]
    companies: list[str]
    results: list[str]
    clean: bool
    result: str

In [51]:
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [94]:
def get_keyword(state: DeepSearch) -> DeepSearch:
    """
    Extracts a single keyword from the user's query that represents the main research topic.
    
    Args:
        state (DeepSearch): The current state containing the user's query
        
    Returns:
        DeepSearch: Updated state with the extracted keyword
        
    The function uses a language model to:
    1. Analyze the user's query
    2. Extract the most relevant single keyword
    3. Convert it to lowercase
    4. Remove any punctuation or quotes
    """
    if state['query'] is None:
        return {
            "keyword": ""
        }
    
    system_prompt_get_keyword = """
        Extract a single lowercase word that captures the core research topic from the user's message.
        Ignore examples or extra context. Return only the topic, with no punctuation or quotes.

        Example:
            Input: "I want to dive deep into how clean energy is evolving…"
            Output: clean energy
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt_get_keyword),
        ("human", "{query}")
    ])
    chain = prompt | llm | StrOutputParser()
    keyword = chain.invoke({"query": state['query']})
    return {
        "keyword": keyword
    }

In [95]:
def get_keywords(state: DeepSearch) -> DeepSearch:
    """
    Extracts multiple relevant keywords from the user's query to better understand their research interests.
    
    Args:
        state (DeepSearch): The current state containing the user's query
        
    Returns:
        DeepSearch: Updated state with the extracted keywords
        
    The function uses a language model to:
    1. Analyze the user's query for main topics and subtopics
    2. Extract 3-5 most relevant keywords
    3. Convert them to lowercase
    4. Return them as a comma-separated string
    
    Example:
        Input: "I'm interested in exploring how AI is transforming healthcare..."
        Output: "ai, healthcare, diagnostics, patient monitoring"
    """

    if state['query'] is None:
        return {
            "keywords": []
        }

    system_prompt_get_keywords = """
        Extract the core keywords from the user's message that indicate the main topic they want to research deeply. Return 3 to 5 keywords in lowercase, in a list. Ignore filler words or phrases.

        Example Input:
            "I'm interested in exploring how AI is transforming healthcare, especially diagnostics and patient monitoring."

        Output:
            ai, healthcare, diagnostics, patient monitoring
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt_get_keywords),
        ("human", "{query}")
    ])
    chain = prompt | llm | CommaSeparatedListOutputParser()
    keywords = chain.invoke({"query": state['query']})
    return {
        "keywords": keywords
    }

In [101]:
def get_companies(state: DeepSearch) -> DeepSearch:
    """
    Identifies and returns the most relevant companies or organizations in the user's research field.
    
    Args:
        state (DeepSearch): The current state containing the user's query and previous analysis
        
    Returns:
        DeepSearch: Updated state with a list of relevant companies
        
    The function uses a language model to:
    1. Analyze the user's research topic
    2. Identify 3-5 leading companies/organizations in the field
    3. Focus on global market leaders and innovators
    4. Return company names as a comma-separated list
    
    Example:
        Input state with query: "artificial intelligence in autonomous vehicles"
        Output state with companies: ["Tesla", "Waymo", "Nvidia", "Cruise", "Mobileye"]
    """

    if state['query'] is None:
        return {
            "companies": []
        }

    system_prompt_get_companies = """
        Based on the user's topic, list 3 to 5 of the most relevant companies or organizations currently leading or innovating in this field. Focus on global relevance and impact. Return only company names, separated by commas, without any other text.

        Example Input:
            Topic: "artificial intelligence in autonomous vehicles"

        Output:
            Tesla, Waymo, Nvidia, Cruise, Mobileye
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt_get_companies),
        ("human", "{query}")
    ])
    chain = prompt | llm | CommaSeparatedListOutputParser()
    companies = chain.invoke({"query": state['query']})
    return {
        "companies": companies
    }

In [102]:
def merge_results(state: DeepSearch) -> DeepSearch:
    """
    Combines all the analysis results into a single state object.
    
    Args:
        state (DeepSearch): The current state containing all individual analysis results
            including query, keyword, keywords list, and companies list
            
    Returns:
        DeepSearch: A complete state object containing all analysis results merged together
        
    The function consolidates:
    - Original query
    - Primary keyword
    - List of related keywords
    - List of relevant companies
    
    Example:
        Input state with separate results
        Output state with all results merged into a single object
    """
    return {
        "query": state['query'],
        "keyword": state['keyword'],
        "keywords": state['keywords'],
        "companies": state['companies']
    }

In [116]:
def clean_input(state: DeepSearch) -> DeepSearch:
    if state['query'] is None:
        return {
            "query": None
        }
    
    system_prompt_clean = """
        Does the user’s message indicate they want to research or explore a specific topic? Answer with true or false only.

        Example Input:
            “I’m curious about how blockchain is used in supply chain management.”,
            "I want to dive deep into how clean energy is evolving…"

        Output:
            true
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt_clean),
        ("human", "{query}")
    ])
    chain =  prompt | llm | StrOutputParser()
    clean = chain.invoke({"query": state['query']})

    if clean == "true":
        return {
            "query": state['query'].lower().strip(),
            "clean": True
        }
    else:
        return {
            "query": None
        }

In [117]:
builder = StateGraph(DeepSearch)

builder.add_node("clean_input", clean_input)
builder.add_node("get_keyword", get_keyword)
builder.add_node("get_keywords", get_keywords)
builder.add_node("get_companies", get_companies)
builder.add_node("merge_results", merge_results)

builder.add_edge(START, "clean_input")

builder.add_edge("clean_input", "get_keyword")
builder.add_edge("clean_input", "get_keywords")
builder.add_edge("clean_input", "get_companies")

builder.add_edge("get_keyword", "merge_results")
builder.add_edge("get_keywords", "merge_results")
builder.add_edge("get_companies", "merge_results")
builder.add_edge("merge_results", END)

graph = builder.compile()

In [ ]:
graph.stream({"query": "Give a deep search about the ICL eye surgery"})

{'query': 'give a deep search about the icl eye surgery',
 'keyword': 'icl eye surgery',
 'keywords': ['icl', 'eye surgery', 'research'],
 'companies': ['Due to the specific nature of your request',
  'I found the following relevant companies in the field of ICL (Implantable Collamer Lens) eye surgery: Staar Surgical Company',
  'Visian ICL',
  'Ophtec',
  'Oculentis',
  'and AcuFocus.']}